# SAGE CIFAR-10 Fresh Colab Run

This notebook assumes the GitHub repo is public and that your latest local code has already been pushed to `main`.

It always deletes the old `/content/superweights--dynamic` folder and reclones from GitHub, so Colab cannot accidentally keep stale code.

## 1. Check GPU

Use `Runtime -> Change runtime type -> T4 GPU` or better before running.

In [ ]:
!nvidia-smi

import torch

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

## 2. Fresh Clone Latest Main

This removes any old checkout and clones the public GitHub repo again.

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/SelbinyyazS/superweights--dynamic.git"
BRANCH = "main"
WORKDIR = Path("/content/superweights--dynamic")

if WORKDIR.exists():
    shutil.rmtree(WORKDIR)

subprocess.run(
    ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(WORKDIR)],
    check=True,
)
os.chdir(WORKDIR)

print("cwd:", os.getcwd())
print("commit:")
subprocess.run(["git", "rev-parse", "--short", "HEAD"], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

train_py = Path("train.py").read_text()
assert "cifar10" in train_py, "This clone is stale: train.py does not contain cifar10 support. Push latest main and rerun this cell."
assert "cifar_cnn" in train_py, "This clone is stale: train.py does not contain cifar_cnn support. Push latest main and rerun this cell."
print("repo ok: CIFAR-10 + cifar_cnn support found")

## 3. Synthetic Smoke Test

No dataset download. This checks masking, SAGE scores, channel pruning, and compaction.

In [ ]:
!python smoke_test.py

## 4. Tiny CIFAR-10 Sanity Run

This downloads CIFAR-10 and runs only a few batches. Use this before spending GPU time on a real sweep.

In [ ]:
!python train.py \
  --dataset cifar10 \
  --model cifar_cnn \
  --dense_start \
  --epochs 1 \
  --base_channels 16 \
  --batch_size 128 \
  --train_batches 10 \
  --eval_batches 5 \
  --num_workers 2 \
  --log_path logs/colab_tiny_cifar10_sanity.csv

## 5. Starter CIFAR-10 Comparison

This is intentionally smaller than the serious run. It compares dense, pure SAGE, activation-only, gradient-only, Taylor, magnitude, and random with one seed. If this works, scale up in the next cell.

In [ ]:
!python run_multiseed.py \
  --dataset cifar10 \
  --model cifar_cnn \
  --modes dense sage_pure activation gradient taylor magnitude random \
  --seeds 1 \
  --epochs 30 \
  --post_compact_epochs 5 \
  --base_channels 32 \
  --batch_size 128 \
  --output_dir logs/cifar10_colab_starter \
  --neuron_prune_start_epoch 10 \
  --neuron_prune_end_epoch 30 \
  --neuron_prune_interval 2 \
  --neuron_prune_fraction 0.03 \
  --sage_focus_start_epoch 10 \
  --sage_grad_boost 1.15 \
  --sage_boost_fraction 0.03 \
  --weak_grad_decay 0.95 \
  --num_workers 2

## 6. Summarize Starter Results

In [ ]:
!python summarize_logs.py --aggregate logs/cifar10_colab_starter/*.csv \
  --output results/cifar10_colab_starter_aggregate.csv
!cat results/cifar10_colab_starter_aggregate.csv

## 7. Bigger Run

Run this only after the starter comparison works. On free Colab, this may take a while.

In [ ]:
# Uncomment and run when ready.
# !python run_multiseed.py \
#   --dataset cifar10 \
#   --model cifar_cnn \
#   --modes dense sage_pure activation gradient taylor magnitude random \
#   --seeds 1 2 3 \
#   --epochs 80 \
#   --post_compact_epochs 20 \
#   --base_channels 64 \
#   --batch_size 128 \
#   --output_dir logs/cifar10_colab_full \
#   --neuron_prune_start_epoch 20 \
#   --neuron_prune_end_epoch 80 \
#   --neuron_prune_interval 2 \
#   --neuron_prune_fraction 0.03 \
#   --sage_focus_start_epoch 20 \
#   --sage_grad_boost 1.15 \
#   --sage_boost_fraction 0.03 \
#   --weak_grad_decay 0.95 \
#   --num_workers 2
#
# !python summarize_logs.py --aggregate logs/cifar10_colab_full/*.csv \
#   --output results/cifar10_colab_full_aggregate.csv
# !cat results/cifar10_colab_full_aggregate.csv

## 8. Visualize SAGE Structure

This renders one SVG map for the SAGE starter run: FLOPs, active channels, and before/after tensor dimensions.

In [ ]:
!python visualize_structure.py logs/cifar10_colab_starter/sage_pure_seed1.csv \
  --output results/cifar10_colab_starter_sage_pure_seed1_structure.svg

## 9. Download Results

In [ ]:
!zip -r sage_cifar10_colab_results.zip logs results

from google.colab import files

files.download("sage_cifar10_colab_results.zip")